# 06 — Filtragem colaborativa item–item

**DATA SCIENCE KNOWLEDGE BASE**  
**Área:** `09_recommender_systems` — Sistemas de Recomendação  
**Capítulo:** 06 de 16  

> Execute as células em sequência, de cima para baixo, em um kernel reiniciado. Use o **Guia Técnico** e a **Cheat Sheet** da seção Materiais complementares como referências para este capítulo.

## Objetivos de aprendizagem
- Comparar vetores de interação por item
- Gerar itens candidatos sem recomendar os já avaliados

## Materiais complementares

- [Guia Técnico](reference_materials/06_similaridade_itens.pdf)
- [Cheat Sheet](cheat_sheets/cheat_sheet_06_similaridade_itens.pdf)

**Navegação:** os links são relativos à pasta deste notebook. A versão editorial desses PDFs será revalidada após a consolidação científica dos notebooks.

## Fundamentação conceitual
Na filtragem item–item, cada filme é representado por seu vetor de usuários. O custo de construir toda a matriz de similaridades cresce quadraticamente no número de itens; recortar itens é adequado para demonstração.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
DATA = Path('datasets')
if not (DATA / 'ratings.csv').exists():
    raise FileNotFoundError('Abra o JupyterLab na pasta da Disciplina 09 (onde existe datasets/).')
ratings = pd.read_csv(DATA / 'ratings.csv')
movies = pd.read_csv(DATA / 'movies.csv')
print('Avaliações:', len(ratings), '| Filmes:', len(movies), '| Usuários:', ratings.userId.nunique())

Avaliações: 100836 | Filmes: 9742 | Usuários: 610


## Matriz e similaridade entre filmes populares

In [2]:
ids=ratings.movieId.value_counts().head(220).index
M=ratings[ratings.movieId.isin(ids)].pivot_table(index='movieId',columns='userId',values='rating').fillna(0)
S=cosine_similarity(M)
query=1
p=M.index.get_loc(query); order=np.argsort(-S[p]); order=order[order!=p][:8]
res=pd.DataFrame({'movieId':M.index[order],'sim':S[p,order]}).merge(movies,on='movieId')
print(res[['title','sim']].to_string(index=False))

                                            title      sim
                               Toy Story 2 (1999) 0.572601
                             Jurassic Park (1993) 0.565637
             Independence Day (a.k.a. ID4) (1996) 0.564262
        Star Wars: Episode IV - A New Hope (1977) 0.557388
                              Forrest Gump (1994) 0.547096
                            Lion King, The (1994) 0.541145
Star Wars: Episode VI - Return of the Jedi (1983) 0.541089
                       Mission: Impossible (1996) 0.538913


**Interpretação.** Semelhança colaborativa é baseada em padrões de usuários, não no gênero.

## Filtrar histórico

In [3]:
seen=set(ratings.loc[ratings.userId.eq(1),'movieId'])
valid=res[~res.movieId.isin(seen)]
print('Candidatos não vistos:',len(valid))

Candidatos não vistos: 2


**Interpretação.** Filtrar itens conhecidos evita recomendações redundantes.

## Aprofundamento e aplicação

**Ponto de atenção deste capítulo:** similaridade de itens. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **similaridade de itens** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **similaridade de itens** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


## Análise crítica — Suporte na similaridade item–item

A semelhança entre filmes resulta dos vetores de avaliações de usuários, não de gêneros. Uma grande similaridade com **poucos coavaliadores** pode ser instável. O recorte pelos filmes mais populares facilita a demonstração, mas limita a cobertura do catálogo e pode reforçar viés de exposição.

**Origem:** aprofundamento técnico autoral fundamentado no experimento deste notebook. Não é transcrição dos materiais históricos.

In [4]:
# Conferir quantos usuários avaliaram cada par sugerido
pares = []
mat_obs = ratings[ratings.movieId.isin(M.index)].pivot_table(index='movieId',columns='userId',values='rating')
for item_id in res.movieId.head(5):
    suporte = int((mat_obs.loc[query].notna() & mat_obs.loc[item_id].notna()).sum())
    pares.append((int(item_id), suporte))
print('Filme recomendado e número de usuários coavaliadores:', pares)

Filme recomendado e número de usuários coavaliadores: [(3114, 81), (480, 132), (780, 122), (260, 134), (356, 154)]


### Critério de leitura

Interprete os resultados como diagnóstico didático. Para comparar modelos de recomendação, defina previamente população, itens elegíveis, janela temporal, sinal de relevância, baseline e métrica; mantenha treino, validação e teste separados.

**Atividade:** registre o efeito de uma mudança de parâmetro e explique uma ameaça à validade do experimento.

## Limitações e boas práticas
A coocorrência é afetada pela popularidade e pelo grau de sobreposição dos usuários.

## Exercícios
Compare a lista item–item com a lista de conteúdo do capítulo 04.

## Soluções comentadas
O capítulo 04 usa atributos dos itens; este capítulo usa avaliações de outros usuários.

## Síntese
Reveja os resultados e compare métodos usando um protocolo de dados e métricas consistente. Não trate avaliações ausentes como avaliações negativas.

### Exportação de resultados

A próxima célula grava os resultados do capítulo em `outputs/`.


In [5]:
# EP_OUTPUTS_09_V1
# Exportação após os experimentos. Executar este notebook desde o início (Run All).
# Arquivos com prefixo do capítulo evitam colisões com outras saídas.
from pathlib import Path
out = Path('outputs')
out.mkdir(parents=True, exist_ok=True)
res[["movieId", "title", "sim"]].rename(columns={"sim": "similaridade_item_item"}).assign(
    item_referencia=query, ja_avaliado_pelo_usuario_1=res.movieId.isin(seen).to_numpy()
).to_csv(out / "06_similares_item_item.csv", index=False)
print('Outputs do capítulo 06 salvos em:', out.resolve())


Outputs do capítulo 06 salvos em: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/09_recommender_systems/outputs
